# WineReview - EDA

## 1. Introduction

#### 1.1 Project Overview:

This analysis explores the wine varieties by continent from the Wine Reviews dataset on Kaggle. 
The goal is to understand the data and discover hidden trends 


#### 1.2 Objective: 

Help stakeholders 
1. Identify top varieties and wineries 
2. Identity good quality wines at low prices

#### 1.3 Tools used:

1. Python: Pandas, SQLite
2. DB: MySQL

In [118]:
## Loading DATA ....

In [22]:
import pandas as pd

In [23]:
df = pd.read_csv('winemag-data_first150k.csv')

In [24]:
df.head()

,Unnamed: 0,country,description,designation,points,price,province,region_1,region_2,variety,winery
0,0,US,This tremendous 100% varietal wine hails from ...,Martha's Vineyard,96,235.0,California,Napa Valley,Napa,Cabernet Sauvignon,Heitz
1,1,Spain,"Ripe aromas of fig, blackberry and cassis are ...",Carodorum Selección Especial Reserva,96,110.0,Northern Spain,Toro,NaN,Tinta de Toro,Bodega Carmen Rodríguez
2,2,US,Mac Watson honors the memory of a wine once ma...,Special Selected Late Harvest,96,90.0,California,Knights Valley,Sonoma,Sauvignon Blanc,Macauley
3,3,US,"This spent 20 months in 30% new French oak, an...",Reserve,96,65.0,Oregon,Willamette Valley,Willamette Valley,Pinot Noir,Ponzi
4,4,France,"This is the top wine from La Bégude, named aft...",La Brûlade,95,66.0,Provence,Bandol,NaN,Provence red blend,Domaine de la Bégude


In [25]:
import sqlite3

In [26]:
conn = sqlite3.connect("wine_db.sqlite")  # This creates a .sqlite file
df.to_sql("wine_reviews", conn, if_exists="replace", index=False)
conn.commit()
conn.close()

In [27]:
!pip uninstall prettytable -y
!pip install prettytable==3.8.0

Found existing installation: prettytable 3.8.0
Uninstalling prettytable-3.8.0:
  Successfully uninstalled prettytable-3.8.0
Defaulting to user installation because normal site-packages is not writeable
  Obtaining dependency information for prettytable==3.8.0 from https://files.pythonhosted.org/packages/25/1e/4c284713b092ec384fad4399452f43f6446ad9aabc9c0b3c3c0920cc53b6/prettytable-3.8.0-py3-none-any.whl.metadata
  Using cached prettytable-3.8.0-py3-none-any.whl.metadata (26 kB)
Using cached prettytable-3.8.0-py3-none-any.whl (27 kB)


In [28]:
%load_ext sql

The sql extension is already loaded. To reload it, use:
  %reload_ext sql


In [29]:
%sql sqlite:///wine_db.sqlite

## 2. Data Exploration

In [41]:
%%sql

select * from wine_reviews LIMIT 5;

 * sqlite:///wine_db.sqlite
Done.


Unnamed: 0,country,description,designation,points,price,province,region_1,region_2,variety,winery
0,US,"This tremendous 100% varietal wine hails from Oakville and was aged over three years in oak. Juicy red-cherry fruit and a compelling hint of caramel greet the palate, framed by elegant, fine tannins and a subtle minty tone in the background. Balanced and rewarding from start to finish, it has years ahead of it to develop further nuance. Enjoy 2022–2030.",Martha's Vineyard,96,235.0,California,Napa Valley,Napa,Cabernet Sauvignon,Heitz
1,Spain,"Ripe aromas of fig, blackberry and cassis are softened and sweetened by a slathering of oaky chocolate and vanilla. This is full, layered, intense and cushioned on the palate, with rich flavors of chocolaty black fruits and baking spices. A toasty, everlasting finish is heady but ideally balanced. Drink through 2023.",Carodorum Selección Especial Reserva,96,110.0,Northern Spain,Toro,None,Tinta de Toro,Bodega Carmen Rodríguez
2,US,"Mac Watson honors the memory of a wine once made by his mother in this tremendously delicious, balanced and complex botrytised white. Dark gold in color, it layers toasted hazelnut, pear compote and orange peel flavors, reveling in the succulence of its 122 g/L of residual sugar.",Special Selected Late Harvest,96,90.0,California,Knights Valley,Sonoma,Sauvignon Blanc,Macauley
3,US,"This spent 20 months in 30% new French oak, and incorporates fruit from Ponzi's Aurora, Abetina and Madrona vineyards, among others. Aromatic, dense and toasty, it deftly blends aromas and flavors of toast, cigar box, blackberry, black cherry, coffee and graphite. Tannins are polished to a fine sheen, and frame a finish loaded with dark chocolate and espresso. Drink now through 2032.",Reserve,96,65.0,Oregon,Willamette Valley,Willamette Valley,Pinot Noir,Ponzi
4,France,"This is the top wine from La Bégude, named after the highest point in the vineyard at 1200 feet. It has structure, density and considerable acidity that is still calming down. With 18 months in wood, the wine has developing an extra richness and concentration. Produced by the Tari family, formerly of Château Giscours in Margaux, it is a wine made for aging. Drink from 2020.",La Brûlade,95,66.0,Provence,Bandol,None,Provence red blend,Domaine de la Bégude


#### 2.1 Total reviews

In [30]:
%%sql
SELECT COUNT(*) FROM wine_reviews;

 * sqlite:///wine_db.sqlite
Done.


COUNT(*)
150930


#### 2.2 Number of countries

In [32]:


%%sql

select count(distinct country) from wine_reviews



 * sqlite:///wine_db.sqlite
Done.


count(distinct country)
48


#### 2.3 Count of reviews per country

In [88]:


%%sql

select country, count(*) as 'Review_counts' from wine_reviews
group by country
order by count(*) desc;


 * sqlite:///wine_db.sqlite
Done.


country,Review_counts
US,62397
Italy,23478
France,21098
Spain,8268
Chile,5816
Argentina,5631
Portugal,5322
Australia,4957
New Zealand,3320
Austria,3057


US-France is not a country. Hence Deleting it

In [51]:
%%sql
delete from wine_reviews where country = 'US-France';

 * sqlite:///wine_db.sqlite
1 rows affected.


[]

#### 2.4 Total Wineries

In [35]:
%%sql
select count( distinct winery) Total_wineries
 from wine_reviews;

 * sqlite:///wine_db.sqlite
Done.


Total_wineries
14810


#### 2.5 Average wine price

In [42]:
%%sql
select round(avg(price)) as Average_price 
from wine_reviews;

 * sqlite:///wine_db.sqlite
Done.


Average_price
33.0


#### 2.6 Wine rating range

In [43]:
%%sql

select points, count(*) as reviews
from wine_reviews
group by points
order by points;

 * sqlite:///wine_db.sqlite
Done.


points,reviews
80,898
81,1502
82,4041
83,6048
84,10708
85,12411
86,15573
87,20747
88,17871
89,12921


Points are between (80 and 100) both inclusive although the range is 0-100 according to source

#### 2.7 Top 5 Countries with most reviews

Top 5 countries with most reviews

In [45]:
%%sql
select country, count(*) reviews
 from wine_reviews
 group by country
 order by reviews desc
 limit  5;

 * sqlite:///wine_db.sqlite
Done.


country,reviews
US,62397
Italy,23478
France,21098
Spain,8268
Chile,5816


#### 2.8 Grouping Countries by reviews
Grouping countries based on top reviews # Categorization using case and CTEs.

In [52]:
%%sql

with cte_t1 as (
select country, count(*) reviews
from wine_reviews
group by country
)
,
cte_t2 as (
select a.* ,
case when reviews < 100 then '<100'
when reviews >= 100 and reviews <1000 then '100 - 999'
when reviews >= 1000 and reviews <5000 then '1000 - 4999'
when reviews >= 5000 and reviews <10000 then '5000 - 9999'
when reviews >= 10000 then '>=10000'
end as GroupsOnReviews
from cte_t1 a
)
select GroupsOnReviews, count(country) countries
from cte_t2
group by GroupsOnReviews
order by  countries desc



 * sqlite:///wine_db.sqlite
Done.


GroupsOnReviews,countries
<100,30
1000 - 4999,5
100 - 999,5
5000 - 9999,4
>=10000,3


Add this info to the dataset for easy querying #
1. Create a table that holds the grouping info. 

In [54]:
%%sql
create table CountryReviewsCount(country varchar(255), GroupsOnReviews varchar(255));

 * sqlite:///wine_db.sqlite
Done.


[]

1.1 update it with the results of categorization  

In [56]:
%%sql
 insert into CountryReviewsCount( country, GroupsOnReviews)
 with cte_t1 as (
 select a.*,
 count(*) over (partition by country) as reviews,
 case 
	when count(*) over (partition by country) < 100 then '<100'
	when count(*) over (partition by country) >= 100 and count(*) over (partition by country) <1000 then '100 - 999'
	when count(*) over (partition by country) >= 1000 and count(*) over (partition by country) <5000 then '1000 - 4999'
	when count(*) over (partition by country) >= 5000 and count(*) over (partition by country) <10000 then '5000 - 9999'
	when count(*) over (partition by country) >= 10000 then '>=10000'
	end as GroupsOnReviews
 from wine_reviews a
 ) 
 select distinct country, GroupsOnReviews
 from cte_t1;

 * sqlite:///wine_db.sqlite
48 rows affected.


[]

2. Update the dataset using left join 

2.1 create new column in wine dataset to hold category data

In [57]:
%%sql
alter table wine_reviews add column GroupsOnReviews varchar(255);

 * sqlite:///wine_db.sqlite
Done.


[]

2.2 Using left join update the wine dataset

In [60]:
%%sql
UPDATE wine_reviews
SET GroupsOnReviews = (
    SELECT b.GroupsOnReviews
    FROM CountryReviewsCount b
    WHERE wine_reviews.country = b.country
);

 * sqlite:///wine_db.sqlite
150929 rows affected.


[]

3. Test the data

In [62]:
%%sql
select distinct country from wine_reviews where GroupsOnReviews = '>=10000';


 * sqlite:///wine_db.sqlite
Done.


country
US
France
Italy


In [63]:
%%sql

select distinct country from wine_reviews where GroupsOnReviews = '100 - 999';


 * sqlite:///wine_db.sqlite
Done.


country
Israel
Greece
Romania
Canada
Hungary


#### 2.9 Adding some metadata to the countries - Continent, lat, lon

In [69]:
%%sql
alter table wine_reviews add  latitude double


 * sqlite:///wine_db.sqlite
Done.


[]

In [67]:
%%sql
alter table wine_reviews add longitude double

 * sqlite:///wine_db.sqlite
Done.


[]

In [68]:
%%sql
alter table wine_reviews add country_code text;

 * sqlite:///wine_db.sqlite
Done.


[]

In [66]:
%%sql
alter table wine_reviews add  column continent text;

 * sqlite:///wine_db.sqlite
Done.


[]

Import a new table containing geo info

In [75]:
df = pd.read_excel('CountryLatLong.xlsx', sheet_name = 'CountryLatLong')

In [76]:
df.head()

,country,latitude,longitude,Country_name,name,Continent
0,AF,33.939110,67.709953,Afghanistan,Afghanistan,Asia
1,AL,41.153332,20.168331,Albania,Albania,Europe
2,DZ,28.033886,1.659626,Algeria,Algeria,Africa
3,AS,-14.270972,-170.132217,American Samoa,American Samoa,Oceania
4,AD,42.546245,1.601554,Andorra,Andorra,Europe


In [81]:
conn = sqlite3.connect("wine_db.sqlite")
df.to_sql("countrylatlong", conn, if_exists="replace", index=False)
conn.commit()
conn.close()

In [82]:
%%sql

select * from wine_reviews LIMIT 5;

 * sqlite:///wine_db.sqlite
Done.


Unnamed: 0,country,description,designation,points,price,province,region_1,region_2,variety,winery,GroupsOnReviews,continent,longitude,country_code,latitude
0,US,"This tremendous 100% varietal wine hails from Oakville and was aged over three years in oak. Juicy red-cherry fruit and a compelling hint of caramel greet the palate, framed by elegant, fine tannins and a subtle minty tone in the background. Balanced and rewarding from start to finish, it has years ahead of it to develop further nuance. Enjoy 2022–2030.",Martha's Vineyard,96,235.0,California,Napa Valley,Napa,Cabernet Sauvignon,Heitz,>=10000,None,None,None,None
1,Spain,"Ripe aromas of fig, blackberry and cassis are softened and sweetened by a slathering of oaky chocolate and vanilla. This is full, layered, intense and cushioned on the palate, with rich flavors of chocolaty black fruits and baking spices. A toasty, everlasting finish is heady but ideally balanced. Drink through 2023.",Carodorum Selección Especial Reserva,96,110.0,Northern Spain,Toro,None,Tinta de Toro,Bodega Carmen Rodríguez,5000 - 9999,None,None,None,None
2,US,"Mac Watson honors the memory of a wine once made by his mother in this tremendously delicious, balanced and complex botrytised white. Dark gold in color, it layers toasted hazelnut, pear compote and orange peel flavors, reveling in the succulence of its 122 g/L of residual sugar.",Special Selected Late Harvest,96,90.0,California,Knights Valley,Sonoma,Sauvignon Blanc,Macauley,>=10000,None,None,None,None
3,US,"This spent 20 months in 30% new French oak, and incorporates fruit from Ponzi's Aurora, Abetina and Madrona vineyards, among others. Aromatic, dense and toasty, it deftly blends aromas and flavors of toast, cigar box, blackberry, black cherry, coffee and graphite. Tannins are polished to a fine sheen, and frame a finish loaded with dark chocolate and espresso. Drink now through 2032.",Reserve,96,65.0,Oregon,Willamette Valley,Willamette Valley,Pinot Noir,Ponzi,>=10000,None,None,None,None
4,France,"This is the top wine from La Bégude, named after the highest point in the vineyard at 1200 feet. It has structure, density and considerable acidity that is still calming down. With 18 months in wood, the wine has developing an extra richness and concentration. Produced by the Tari family, formerly of Château Giscours in Margaux, it is a wine made for aging. Drink from 2020.",La Brûlade,95,66.0,Provence,Bandol,None,Provence red blend,Domaine de la Bégude,>=10000,None,None,None,None


In [83]:
%%sql

select * from countrylatlong LIMIT 5;

 * sqlite:///wine_db.sqlite
Done.


country,latitude,longitude,Country_name,name,Continent
AF,33.93911,67.709953,Afghanistan,Afghanistan,Asia
AL,41.153332,20.168331,Albania,Albania,Europe
DZ,28.033886,1.659626,Algeria,Algeria,Africa
AS,-14.270972,-170.132217,American Samoa,American Samoa,Oceania
AD,42.546245,1.601554,Andorra,Andorra,Europe


In [84]:
%%sql 

UPDATE wine_reviews
SET 
    country_code = (
        SELECT b.country
        FROM countrylatlong b
        WHERE b.country_name = wine_reviews.country
    ),
    continent = (
        SELECT b.continent
        FROM countrylatlong b
        WHERE b.country_name = wine_reviews.country
    ),
    latitude = (
        SELECT b.latitude
        FROM countrylatlong b
        WHERE b.country_name = wine_reviews.country
    ),
    longitude = (
        SELECT b.longitude
        FROM countrylatlong b
        WHERE b.country_name = wine_reviews.country
    );

 * sqlite:///wine_db.sqlite
150929 rows affected.


[]

In [85]:
%%sql

select * from wine_reviews LIMIT 5;

 * sqlite:///wine_db.sqlite
Done.


Unnamed: 0,country,description,designation,points,price,province,region_1,region_2,variety,winery,GroupsOnReviews,continent,longitude,country_code,latitude
0,US,"This tremendous 100% varietal wine hails from Oakville and was aged over three years in oak. Juicy red-cherry fruit and a compelling hint of caramel greet the palate, framed by elegant, fine tannins and a subtle minty tone in the background. Balanced and rewarding from start to finish, it has years ahead of it to develop further nuance. Enjoy 2022–2030.",Martha's Vineyard,96,235.0,California,Napa Valley,Napa,Cabernet Sauvignon,Heitz,>=10000,North America,-95.712891,US,37.09024
1,Spain,"Ripe aromas of fig, blackberry and cassis are softened and sweetened by a slathering of oaky chocolate and vanilla. This is full, layered, intense and cushioned on the palate, with rich flavors of chocolaty black fruits and baking spices. A toasty, everlasting finish is heady but ideally balanced. Drink through 2023.",Carodorum Selección Especial Reserva,96,110.0,Northern Spain,Toro,None,Tinta de Toro,Bodega Carmen Rodríguez,5000 - 9999,Europe,-3.74922,ES,40.463667
2,US,"Mac Watson honors the memory of a wine once made by his mother in this tremendously delicious, balanced and complex botrytised white. Dark gold in color, it layers toasted hazelnut, pear compote and orange peel flavors, reveling in the succulence of its 122 g/L of residual sugar.",Special Selected Late Harvest,96,90.0,California,Knights Valley,Sonoma,Sauvignon Blanc,Macauley,>=10000,North America,-95.712891,US,37.09024
3,US,"This spent 20 months in 30% new French oak, and incorporates fruit from Ponzi's Aurora, Abetina and Madrona vineyards, among others. Aromatic, dense and toasty, it deftly blends aromas and flavors of toast, cigar box, blackberry, black cherry, coffee and graphite. Tannins are polished to a fine sheen, and frame a finish loaded with dark chocolate and espresso. Drink now through 2032.",Reserve,96,65.0,Oregon,Willamette Valley,Willamette Valley,Pinot Noir,Ponzi,>=10000,North America,-95.712891,US,37.09024
4,France,"This is the top wine from La Bégude, named after the highest point in the vineyard at 1200 feet. It has structure, density and considerable acidity that is still calming down. With 18 months in wood, the wine has developing an extra richness and concentration. Produced by the Tari family, formerly of Château Giscours in Margaux, it is a wine made for aging. Drink from 2020.",La Brûlade,95,66.0,Provence,Bandol,None,Provence red blend,Domaine de la Bégude,>=10000,Europe,2.213749,FR,46.227638


In [86]:
%%sql
 select  country, count(*)  from wine_reviews
 where country_code is null
 group by country;

 * sqlite:///wine_db.sqlite
Done.


country,count(*)
None,5


In [87]:
%%sql

select * from wine_reviews where country_code is null

 * sqlite:///wine_db.sqlite
Done.


Unnamed: 0,country,description,designation,points,price,province,region_1,region_2,variety,winery,GroupsOnReviews,continent,longitude,country_code,latitude
1133,None,"Delicate white flowers and a spin of lemon peel on the nose start this refined white. The bright fruit on the palate is tropical and exotic, but the minerality gives it lift. Fuller-bodied but poised, the wine has aging potential and a food-friendly character.",Askitikos,90,17.0,None,None,None,Assyrtiko,Tsililis,None,None,None,None,None
1440,None,"A blend of 60% Syrah, 30% Cabernet Sauvignon and 10% Merlot, this inky garnet-colored wine offers aromas of cassis and elderberry. On the palate, there is a combination of cooked fruit and cool spice flavors yet there is no lack of accompanying acidity. Flavors of cassis, elderberry, anise, orange peel and vanilla are backed by silky tannins that stay smooth into the cooling finish.",Shah,90,30.0,None,None,None,Red Blend,Büyülübağ,None,None,None,None,None
68226,None,"From first sniff to last, the nose never makes much of an impression; the wine has funk and generic chemical aromas but also your basic red apple and char. Pretty big and aggressive in the mouth, with snaggy acids and rough tannins.",Piedra Feliz,81,15.0,None,None,None,Pinot Noir,Chilcas,None,None,None,None,None
113016,None,"From first sniff to last, the nose never makes much of an impression; the wine has funk and generic chemical aromas but also your basic red apple and char. Pretty big and aggressive in the mouth, with snaggy acids and rough tannins.",Piedra Feliz,81,15.0,None,None,None,Pinot Noir,Chilcas,None,None,None,None,None
135696,None,"From first sniff to last, the nose never makes much of an impression; the wine has funk and generic chemical aromas but also your basic red apple and char. Pretty big and aggressive in the mouth, with snaggy acids and rough tannins.",Piedra Feliz,81,15.0,None,None,None,Pinot Noir,Chilcas,None,None,None,None,None


Delete the 5 rows where country info is not available in the dataset. 

In [90]:
%%sql

delete from wine_reviews where country is null

 * sqlite:///wine_db.sqlite
5 rows affected.


[]

In [91]:
%%sql
 select  country, count(*)  from wine_reviews
 where country_code is null
 group by country;

 * sqlite:///wine_db.sqlite
Done.


country,count(*)


#### 2.10 Reviews by Continent

In [93]:
%%sql

select  continent, count(*)  
from wine_reviews
group by continent
order by count(*) desc

 * sqlite:///wine_db.sqlite
Done.


continent,count(*)
Europe,65416
North America,62656
South America,11564
Oceania,8277
Africa,2275
Asia,736


This dataset is heavily focused on North America and Europe.

In [98]:
%%sql 

WITH cte_t1 AS (
  SELECT 
    continent, 
    COUNT(*) AS records
  FROM wine_reviews
  GROUP BY continent
),
cte_t2 AS (
  SELECT 
    *, 
    SUM(records) OVER () AS total_reviews
  FROM cte_t1
)
SELECT 
  continent, 
  records,
  total_reviews,
  ROUND(1.0 * records / total_reviews * 100, 2) AS percentage
FROM cte_t2
ORDER BY records DESC;

 * sqlite:///wine_db.sqlite
Done.


continent,records,total_reviews,percentage
Europe,65416,150924,43.34
North America,62656,150924,41.51
South America,11564,150924,7.66
Oceania,8277,150924,5.48
Africa,2275,150924,1.51
Asia,736,150924,0.49


#### Note: 
Europe & North American wine reviews form **85%** of the dataset. **Hence only NA & EU** will be analysed in this project

#### 2.11 Price per bottle

In [104]:
%%sql
 select continent, round(avg(price),2) as avg_price, round(max(price),0) max_price, round(min(price),0) min_price
 from wine_reviews 
 where continent in ('Europe', 'North America')
 group by continent
 order by avg(price)

 * sqlite:///wine_db.sqlite
Done.


continent,avg_price,max_price,min_price
North America,33.65,2013.0,4.0
Europe,36.61,2300.0,4.0


##### Note:
1. The average price of wine in North America is greater than that of Europe. 
2. But the expensive variety is from Europe and costs 300 dollars more

#### 2.12 Most Expensive Variety

In [110]:
%%sql

with cte_t1 as (
 select continent, avg(price), max(price) max_price, min(price) min_price
 from wine_reviews 
 where continent in ('Europe', 'North America')
 group by continent
)

select  continent, country, province, winery, variety, round(price,0) as price from 
 wine_reviews where price in (select max_price from cte_t1) 

 * sqlite:///wine_db.sqlite
Done.


continent,country,province,winery,variety,price
North America,US,California,Blair,Chardonnay,2013.0
Europe,France,Bordeaux,Château Latour,Bordeaux-style Red Blend,2300.0


##### Note:
The most expensive bottles are from
1. France Chateau Latour Winery, a type of Red Wine
2. California, USA a variety called Chardonnay

#### 2.13 Popular Wine varieties 

Varities with **more than 1000** reviews

In [116]:
%%sql
select variety, round(avg(price),2) as avg_price, max(price), min(price), count(*) as Reviews
from wine_reviews 
where continent in ('Europe', 'North America')
group by variety 
having count(*) > 1000
order by avg(price) desc

 * sqlite:///wine_db.sqlite
Done.


variety,avg_price,max(price),min(price),Reviews
Champagne Blend,81.68,505.0,7.0,1186
Nebbiolo,66.44,495.0,14.0,2238
Sangiovese Grosso,60.61,900.0,12.0,1346
Port,52.27,980.0,11.0,1038
Bordeaux-style Red Blend,49.77,2300.0,7.0,6852
Cabernet Sauvignon,48.51,625.0,4.0,9703
"Corvina, Rondinella, Molinara",46.46,535.0,8.0,1682
Pinot Noir,46.11,740.0,5.0,12704
Syrah,37.72,450.0,5.0,5082
Bordeaux-style White Blend,36.74,1000.0,8.0,1255


## 3.Summary



1. The most expensive varieties are Bordeaux-style Red Wine & Chardonnay
2. Europe & NA are home to most of the wine varieties in the dataset.
3. The most expensive wine is at 2300 dollars from Château Latour winery in Bordeaux in France

#### The rest of the analysis is continued on separate md files in the same folder. 